In [2]:
import os
from tqdm import tqdm
from glob import glob

import numpy as np
import pandas as pd

from datasets import Dataset, DatasetDict, load_metric

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments,
                          DataCollatorForSeq2Seq, Trainer)
from peft import LoraConfig, get_peft_model, TaskType

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

In [3]:
class CFG:
    wandb = True
    report_to = None
    lab_assignment = 3
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    prefix_val = "summarize: "
    output_dir = "processed_data"
    model_save_dir = "PEFT_T5"

    tokenizer_name = "google/t5-efficient-mini"
    model_name = "google/t5-efficient-mini"

    project = 'NUM-Machine-Learning-Lab-3'
    name = "Lab 3 Model Training - Text Summarization T5 with ROUGE-1"

    config = {
        "output_dir": "t5_small_lab3_finetune_PEFT",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        'num_train_epochs': 3,
        "train_batch_size": 12,
        "eval_batch_size": 4,
        "max_seq_length": 1024,
        "overwrite_output_dir": True,
        "reprocess_input_data": True,
        "fp16": True
    }

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "rouge"

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

# 1. Өгөгдлөө модел сургахад бэлтгэх

In [4]:
df_names = glob("../Lab 2/processed_dfs/*.parquet")
search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

df = pd.DataFrame()
for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)
df['prefix'] = CFG.prefix_val

os.makedirs(CFG.output_dir, exist_ok = True)
output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation.parquet")
df.to_parquet(output_filename)

100%|██████████| 12/12 [00:00<00:00, 17.83it/s]


In [5]:
print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")
print(f"All search terms:\n{search_terms}")

print(df.head())


Dataframe memory usage
Index               132
target_text     1966107
input_text     17616445
prefix           983416
dtype: int64
Dataframe shape: (14462, 3)

All search terms:
audio+classification
audio+deep+learning
audio+encoding
audio+fast+fourier
audio+fourier
audio+generation
audio+machine+learning
audio+prediction
audio+recognition
audio+representation
audio+restoration
audio+signal
                                         target_text  \
0  Improved Mispronunciation detection system usi...   
1  Improving Factored Hybrid HMM Acoustic Modelin...   
2  Disentangling Style and Speaker Attributes for...   
3  Synthetic speech detection using meta-learning...   
4  A Pre-trained Audio-Visual Transformer for Emo...   

                                          input_text       prefix  
0  This report proposes state-of-the-art research...  summarize:   
1  In this work, we show that a factored hybrid h...  summarize:   
2  End-to-end neural TTS has shown improved perfo...  summarize

In [6]:
test_size = CFG.test_size
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation_dataset")
arxiv_title_dict.save_to_disk(output_filename)

print(arxiv_title_dict)

Training instance count: 11570
Test instance count: 2892



Saving the dataset (0/1 shards):   0%|          | 0/11570 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2892 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 11570
    })
    test: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 2892
    })
})


In [7]:
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name, use_fast = False)

def preprocess_function(examples):
    inputs = [CFG.prefix_val + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs,
                             max_length = config['max_seq_length'],
                             padding = True,
                             truncation = True)

    labels = tokenizer(text_target = examples["target_text"],
                       max_length = config["max_seq_length"] // 4,
                       padding = True,
                       truncation = True)
    
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_arxiv = arxiv_title_dict.map(preprocess_function, batched = True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/11570 [00:00<?, ? examples/s]

Map:   0%|          | 0/2892 [00:00<?, ? examples/s]

In [8]:
rouge = load_metric(CFG.eval_metric)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, 
                                           skip_special_tokens = True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, 
                                            skip_special_tokens = True)

    # Compute ROUGE-1 scores
    rouge_scores = rouge.compute(predictions = decoded_preds, 
                                 references = decoded_labels, 
                                 rouge_types = ["rouge1"])["rouge1"]

    # Calculate the mean ROUGE-1 F1 score
    rouge1_f1 = np.mean([score["f"] for score in rouge_scores])

    # Rounds the result to 4 decimal places for cleaner output, and returns it.
    return {"rouge1_f1": round(rouge1_f1, 4)}

# 2. Model Fine-Tuning

In [9]:
lora_config = LoraConfig(
    task_type = TaskType.SEQ_2_SEQ_LM, 
    inference_mode = False, 
    r = 8, 
    lora_alpha = 32, 
    lora_dropout = 0.3
)

model = AutoModelForSeq2SeqLM.from_pretrained(CFG.model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model = CFG.model_name)

peft_model = get_peft_model(model, lora_config)

peft_model.print_trainable_parameters()

trainable params: 172,032 || all params: 31,392,512 || trainable%: 0.548003294543616


In [10]:
training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = config["output_dir"],
    num_train_epochs = config["num_train_epochs"],
    per_device_train_batch_size = config["train_batch_size"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    # Evaluation
    do_eval = False
)

trainer = Trainer(
    model = peft_model,
    args = training_args,
    train_dataset = tokenized_arxiv["train"],
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [11]:
trainer.train()

test_ver = 5
model_save_path = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")
os.makedirs(model_save_path, exist_ok = True)

trainer.model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

  0%|          | 0/2895 [00:00<?, ?it/s]

{'loss': 13.0494, 'grad_norm': 3.169241189956665, 'learning_rate': 1.6545768566493957e-05, 'epoch': 0.52}
{'loss': 9.6705, 'grad_norm': 5.440330982208252, 'learning_rate': 1.3091537132987911e-05, 'epoch': 1.04}
{'loss': 5.0668, 'grad_norm': 3.224266529083252, 'learning_rate': 9.637305699481867e-06, 'epoch': 1.55}
{'loss': 3.1337, 'grad_norm': 1.6756482124328613, 'learning_rate': 6.183074265975821e-06, 'epoch': 2.07}
{'loss': 2.8634, 'grad_norm': 1.7811992168426514, 'learning_rate': 2.728842832469776e-06, 'epoch': 2.59}
{'train_runtime': 273.448, 'train_samples_per_second': 126.935, 'train_steps_per_second': 10.587, 'train_loss': 6.218179916882556, 'epoch': 3.0}


('PEFT_T5/test_v5/tokenizer_config.json',
 'PEFT_T5/test_v5/special_tokens_map.json',
 'PEFT_T5/test_v5/spiece.model',
 'PEFT_T5/test_v5/added_tokens.json')

# 3. Evaluation

In [48]:
import torch

peft_model.eval()

with torch.no_grad():
    outputs = peft_model.generate(input_ids = torch.tensor(tokenized_arxiv['test']['input_ids']), max_new_tokens = 10)
    output_text = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens = True)
print(output_text)

ValueError: expected sequence of length 437 at dim 1 (got 494)

In [52]:
idx = 0
text = "summarize: " + tokenized_arxiv['test'][idx]['input_text']
actual_text = tokenized_arxiv['test'][idx]['target_text']

tmp = tokenizer(text, return_tensors = 'pt')

In [60]:
peft_model = peft_model.to("cuda")

peft_model.eval()

with torch.no_grad():
    outputs = peft_model.generate(input_ids = torch.tensor(tokenized_arxiv['test']['input_ids'][0][0]).to("cuda"), max_new_tokens = 10)
    output_text = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens = True)
print(output_text)

IndexError: tuple index out of range